In [1]:
!pip install -q langchain-text-splitters chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 121.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

document_text = """Database Systems

A database management system provides software for storing, organizing, and retrieving structured information. Tables contain rows and columns that represent related data.

SQL is commonly used to create, read, update, and delete records. Indexes can accelerate queries, while transactions help maintain consistency during operations.

Modern applications may combine relational databases with caches, search engines, and vector databases depending on their requirements."""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=240,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(document_text)
print("Number of chunks:", len(chunks))


Number of chunks: 3


In [3]:
chunk_ids = [f"db_chunk_{i}" for i in range(len(chunks))]

metadata = [
    {
        "source": "database_notes.txt",
        "chunk_index": i,
        "length": len(chunk),
        "category": "database"
    }
    for i, chunk in enumerate(chunks)
]

for cid, meta in zip(chunk_ids, metadata):
    print(cid, meta)


db_chunk_0 {'source': 'database_notes.txt', 'chunk_index': 0, 'length': 189, 'category': 'database'}
db_chunk_1 {'source': 'database_notes.txt', 'chunk_index': 1, 'length': 161, 'category': 'database'}
db_chunk_2 {'source': 'database_notes.txt', 'chunk_index': 2, 'length': 135, 'category': 'database'}


In [4]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()

embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.get_or_create_collection(
    name="database_knowledge",
    embedding_function=embedder
)

print("Collection:", collection.name)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Collection: database_knowledge


In [5]:
collection.add(
    ids=chunk_ids,
    documents=chunks,
    metadatas=metadata
)

print("Stored chunks:", collection.count())


Stored chunks: 3


In [6]:
stored = collection.get(
    include=["documents", "metadatas"]
)

for cid, doc, meta in zip(
    stored["ids"], stored["documents"], stored["metadatas"]
):
    print(f"ID: {cid}")
    print(f"Metadata: {meta}")
    print(f"Text: {doc}\n")


ID: db_chunk_0
Metadata: {'source': 'database_notes.txt', 'chunk_index': 0, 'category': 'database', 'length': 189}
Text: Database Systems

A database management system provides software for storing, organizing, and retrieving structured information. Tables contain rows and columns that represent related data.

ID: db_chunk_1
Metadata: {'length': 161, 'source': 'database_notes.txt', 'category': 'database', 'chunk_index': 1}
Text: SQL is commonly used to create, read, update, and delete records. Indexes can accelerate queries, while transactions help maintain consistency during operations.

ID: db_chunk_2
Metadata: {'chunk_index': 2, 'length': 135, 'category': 'database', 'source': 'database_notes.txt'}
Text: Modern applications may combine relational databases with caches, search engines, and vector databases depending on their requirements.



In [7]:
query = "How do database indexes improve query performance?"

results = collection.query(
    query_texts=[query],
    n_results=2,
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print("Distance:", round(distance, 4))
    print("Source:", meta["source"])
    print("Text:", doc)
    print()


Distance: 0.362
Source: database_notes.txt
Text: SQL is commonly used to create, read, update, and delete records. Indexes can accelerate queries, while transactions help maintain consistency during operations.

Distance: 0.5765
Source: database_notes.txt
Text: Modern applications may combine relational databases with caches, search engines, and vector databases depending on their requirements.



In [8]:
second_text = """Data Analytics

Data analytics transforms raw information into useful insights. Descriptive analytics summarizes what happened, while predictive analytics estimates what may happen next.

Visualization tools can present trends using charts and dashboards. Good analytics workflows depend on clean data, appropriate metrics, and clear interpretation."""

second_chunks = splitter.split_text(second_text)

collection.add(
    ids=[f"analytics_{i}" for i in range(len(second_chunks))],
    documents=second_chunks,
    metadatas=[
        {
            "source": "analytics_notes.txt",
            "chunk_index": i,
            "category": "analytics"
        }
        for i in range(len(second_chunks))
    ]
)

print("Total chunks after second document:", collection.count())


Total chunks after second document: 5


In [9]:
results = collection.query(
    query_texts=["What is predictive analytics?"],
    n_results=2,
    where={"source": "analytics_notes.txt"},
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(distance, meta, doc)


0.42192739248275757 {'category': 'analytics', 'source': 'analytics_notes.txt', 'chunk_index': 0} Data Analytics

Data analytics transforms raw information into useful insights. Descriptive analytics summarizes what happened, while predictive analytics estimates what may happen next.
0.5937936902046204 {'category': 'analytics', 'chunk_index': 1, 'source': 'analytics_notes.txt'} Visualization tools can present trends using charts and dashboards. Good analytics workflows depend on clean data, appropriate metrics, and clear interpretation.
